# Extracción base de datos astronomía
Con este cuaderno se pretende crear una base de datos sobre astronomía.
Contiene:


*   Constelaciones
*   Estrellas
*   Planetas
*   Planetas enanos
*   Galaxias



### Instalar

In [ ]:
!pip install astroquery

In [ ]:
!pip install SPARQLWrapper

In [ ]:
!pip install wikipedia

In [ ]:
import pandas as pd
import requests
import io
import re
from bs4 import BeautifulSoup
from astroquery.vizier import Vizier
import warnings
from urllib.parse import unquote
import time

## Sacar las tablas por separado

Extraer la tabla de la Wikipedia y generaar una tabla de **costelaciones** que tenga por columnas: Abreviatura, Nombre_Es, Nombre_Latin, Categoría, Contenido, Num_estrellas, Brillo, Coordenadas, Ubicacion y ULR_Wikipedia.


In [ ]:
def limpiar(texto): # Función para limpiar el texto y quitar las referencias de la wiki
    texto = str(texto)
    texto = re.sub(r'\[.*?\]|\(.*?\)', '', texto)
    return texto.strip()

url = "https://es.wikipedia.org/wiki/Anexo:Constelaciones" # Enlace wiki
respuesta = requests.get(url)
time.sleep(5) # Tiempo de espera para no saturar la Wikipedia

if respuesta.status_code == 200: # Accedimos bien a la wikipedia
    print("Se ha accedido a la Wikipedia correctamente")
    soup = BeautifulSoup(respuesta.text, 'html.parser')

    tabla_html = next(t for t in soup.find_all('table') if len(t.find_all('tr')) >= 88) # Cogemos la tabla q tiene al menos 88 pq 88 son las constelaciones oficilaes

    datos_lista = []
    filas = tabla_html.find_all('tr')[1:] # Saltamos la cabecera q es el 0

    for fila in filas:
        celdas = fila.find_all(['td', 'th'])
        if len(celdas) >= 3:
            # Extraer nombres y abreviaturas
            nombre_latin = limpiar(celdas[0].get_text())
            nombre_es = limpiar(celdas[1].get_text())
            abreviatura = limpiar(celdas[2].get_text())

            # Extraer el enlace
            enlace_encontrado = fila.find('a', href=re.compile(r'^/wiki/(?!Archivo:|File:|Special:)'))

            if enlace_encontrado:
                url_1 = "https://es.wikipedia.org" + enlace_encontrado['href']
                url_2 = unquote(url_1)
            else:
                url_2 = "Sin enlace"

            datos_lista.append({
                'Nombre_Latin': nombre_latin,
                'Nombre': nombre_es,
                'ID': abreviatura,
                'URL_Wikipedia': url_2,
                'Categoría': 'Constelación',
            })

    # Convertimos a DataFrame de Pandas
    df = pd.DataFrame(datos_lista)
    # Sacar los datos de VizieR y las introducciones de wikipedia
    print("Buscando contexto en VizieR y Wikipedia")

    def contexto(fila):
        # Sacar introducción de Wikipedia
        contenido = "Descripción no disponible." # Así si de alguna no obtenemos info quedará esto y lo podemos revisar
        if fila['URL_Wikipedia'] != "Sin enlace":
            try:
                # Entramos a la página individual
                res_p = requests.get(fila['URL_Wikipedia'], timeout=5)
                time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
                soup_p = BeautifulSoup(res_p.text, 'html.parser')

                # Buscamos el primer párrafo que tenga texto sustancioso
                for p in soup_p.find_all('p'):
                    texto_sucio = p.get_text().strip()
                    # Si el párrafo tiene más de 100 caracteres, es nuestra introducción
                    if len(texto_sucio) > 100:
                        contenido = limpiar(texto_sucio)
                        break
            except:
                pass

        # Eliminamos la consuulta a VizieR en constelaciones pq daba resultados imprecisos
        coords, brillo, ubicacion = "n/a", "n/a", "n/a"

        return pd.Series([contenido, coords, brillo, ubicacion])

    # Aplicamos la función a cada fila
    df[['Contenido', 'Coordenadas', 'Brillo', 'Ubicacion']] = df.apply(contexto, axis=1)

    # Reordenar
    columnas_finales = [
        'ID', 'Nombre', 'Nombre_Latin', 'Categoría',
        'Contenido', 'Brillo', 'Coordenadas', 'Ubicacion', 'URL_Wikipedia'
    ]
    df_final = df[columnas_finales]

    # Guardamos el archivo
    df_final.to_csv('constelaciones_completa.csv', index=False, encoding='utf-8-sig')
    print("Porceso finalizado exitosamente")
else:
    print(f"Error al acceder a la Wikiedia. Código de error: {respuesta.status_code}")

Se ha accedido a la Wikipedia correctamente
Buscando contexto en VizieR y Wikipedia
Porceso finalizado exitosamente


Ahora hacemos lo mismo pero con las **estrellas**

In [ ]:
def limpiar(texto): # Función para limpiar el texto y quitar las referencias de la wiki
    texto = str(texto)
    texto = re.sub(r'\[.*?\]|\(.*?\)', '', texto)
    return texto.strip()

url_principal = "https://es.wikipedia.org/wiki/Anexo:Estrellas"

respuesta = requests.get(url_principal)
time.sleep(5) # Tiempo de espera para no saturar la Wikipedia

if respuesta.status_code == 200:
    print("Se ha accedido a la Wikipedia correctamente")
    soup_principal = BeautifulSoup(respuesta.text, 'html.parser')

    tablas = soup_principal.find_all('table', class_='wikitable')
    print(f"Se han encontrado {len(tablas)} tablas de costelaciones (una por letra).")

    datos_estrellas = []

    for i, tabla in enumerate(tablas):
        filas = tabla.find_all('tr')[1:] # Saltamos el encabezado

        for fila in filas:
            celdas = fila.find_all(['td', 'th'])
            # En este anexo: Col 0 = Nombre, Col 1 = Designación, Col 2 = Constelación
            if len(celdas) >= 2:
                nombre_estrella = limpiar(celdas[0].get_text())
                designacion = limpiar(celdas[1].get_text())
                constelacion = limpiar(celdas[2].get_text())

                # Extraer el enlace
                tag_a = celdas[0].find('a', href=re.compile(r'^/wiki/(?!Archivo:|File:|Special:)'))
                if tag_a:
                    url_estrella = unquote("https://es.wikipedia.org" + tag_a['href'])
                else:
                    url_estrella = "Sin enlace"

                datos_estrellas.append({
                    'Nombre': nombre_estrella,
                    'ID' : designacion,
                    'Constelacion': constelacion,
                    'URL_Wikipedia': url_estrella,
                    'Categoría': 'Estrella'
                })

    df = pd.DataFrame(datos_estrellas)
    print(f"Total de estrellas encontradas: {len(df)}")

    # Wiki y VizieR
    def enriquecer_estrella(fila):
        resumen = "Descripción no disponible."
        coords, brillo, ubicacion = "n/a", "n/a", "n/a" # Se inicializan con 'n/a'

        # Sacar introducción de Wikipedia
        if fila['URL_Wikipedia'] != "Sin enlace":
            try:
                res_p = requests.get(fila['URL_Wikipedia'], timeout=5)
                time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
                if res_p.status_code == 200:
                    soup_p = BeautifulSoup(res_p.text, 'html.parser')
                    cuerpo = soup_p.find('div', class_='mw-parser-output')
                    if cuerpo:
                        for p in cuerpo.find_all('p', recursive=False):
                            txt = p.get_text().strip()
                            if len(txt) > 80: # Párrafo mínimo
                                resumen = limpiar(txt)
                                break
            except:
                pass

        # Las consultas a VizieR se han eliminado, los valores serán 'n/a' por defecto

        return pd.Series([resumen, coords, brillo, ubicacion])

    print("Extrayendo intormación...")
    # Solo los 10 primeros para que tarde menos
    # df[['Contenido', 'Coordenadas', 'Brillo', 'Ubicacion']] = df.head(10).apply(enriquecer_estrella, axis=1)
    # Tabla completa
    df[['Contenido', 'Coordenadas', 'Brillo', 'Ubicacion']] = df.apply(enriquecer_estrella, axis=1)

    columnas_finales = [
        'ID', 'Nombre', 'Constelacion', 'Categoría',
        'Contenido', 'Brillo', 'Coordenadas', 'Ubicacion', 'URL_Wikipedia'
    ]
    df_final = df[columnas_finales]

    # Guardamos el archivo
    df.to_csv('estrellas_completo.csv', index=False, encoding='utf-8-sig')
    print("Porceso finalizado exitosamente")

else:
    print(f"Error al acceder a la Wikiedia. Código de error: {respuesta.status_code}")

Se ha accedido a la Wikipedia correctamente
Se han encontrado 25 tablas de costelaciones (una por letra).
Total de estrellas encontradas: 448
Extrayendo intormación...


/usr/local/lib/python3.12/dist-packages/astroquery/vizier/core.py:827: UserWarning: VOTABLE parsing raised exception: 26:33: not well-formed (invalid token)
  warnings.warn("VOTABLE parsing raised exception: {0}".format(ex))
/usr/local/lib/python3.12/dist-packages/astroquery/vizier/core.py:827: UserWarning: VOTABLE parsing raised exception: 26:30: not well-formed (invalid token)
  warnings.warn("VOTABLE parsing raised exception: {0}".format(ex))
/usr/local/lib/python3.12/dist-packages/astroquery/vizier/core.py:827: UserWarning: VOTABLE parsing raised exception: 26:29: not well-formed (invalid token)
  warnings.warn("VOTABLE parsing raised exception: {0}".format(ex))
/usr/local/lib/python3.12/dist-packages/astroquery/vizier/core.py:827: UserWarning: VOTABLE parsing raised exception: 26:33: not well-formed (invalid token)
  warnings.warn("VOTABLE parsing raised exception: {0}".format(ex))
/usr/local/lib/python3.12/dist-packages/astroquery/vizier/core.py:827: UserWarning: VOTABLE parsing 

Porceso finalizado exitosamente


Repetimos el proceso con el anexo de los planetas del sistema solar

In [ ]:
def arreglar(celda): # Arregla problemas con la notacion científica
    for sup in celda.find_all('sup'):
        contenido_sup = sup.get_text().strip()
        if contenido_sup.isdigit():
            sup.replace_with(f"E{contenido_sup}")

    texto = celda.get_text()
    texto = re.sub(r'[×·]\s*10\s*E', 'E', texto)
    return texto

def limpiar(texto):
    texto = str(texto)
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.replace('\xa0', ' ')
    return texto.strip()

# Como la tabal anexo de la wikipedia es dificil de leer y son pocos planetas, metemos los enlaces a mano
enlaces_planetas = {
    "Mercurio": "https://es.wikipedia.org/wiki/Mercurio_(planeta)",
    "Venus": "https://es.wikipedia.org/wiki/Venus_(planeta)",
    "Tierra": "https://es.wikipedia.org/wiki/Tierra",
    "Marte": "https://es.wikipedia.org/wiki/Marte_(planeta)",
    "Júpiter": "https://es.wikipedia.org/wiki/J%C3%BApiter_(planeta)",
    "Saturno": "https://es.wikipedia.org/wiki/Saturno_(planeta)",
    "Urano": "https://es.wikipedia.org/wiki/Urano_(planeta)",
    "Neptuno": "https://es.wikipedia.org/wiki/Neptuno_(planeta)"
}

def extraer_datos_planeta(nombre, url):
    resumen = "Descripción no disponible."
    datos_tecnicos = {"Masa": "n/a", "Gravedad": "n/a", "Diámetro": "n/a"}

    try:
        r = requests.get(url, timeout=10)
        time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, 'html.parser')

            # Extraer el Contenido (Primer párrafo largo)
            for p in soup.find_all('p'):
                texto_p = p.get_text().strip()
                if len(texto_p) > 120:
                    resumen = limpiar(texto_p)
                    break

            # Extraer Datos Técnicos de la 'infobox' que es la ficha lateral de las páginas de la wiki
            infobox = soup.find('table', class_='infobox')
            if infobox:
                filas_info = infobox.find_all('tr')
                for f in filas_info:
                    texto_fila = f.get_text().lower()

                    if 'masa' in texto_fila and datos_tecnicos["Masa"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Masa"] = limpiar(arreglar(valor))

                    if 'gravedad' in texto_fila and datos_tecnicos["Gravedad"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Gravedad"] = limpiar(arreglar(valor))

                    if 'diámetro' in texto_fila and datos_tecnicos["Diámetro"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Diámetro"] = limpiar(arreglar(valor))

    except Exception as e:
        print(f"Error con {nombre}: {e}")

    return {
        "Nombre": nombre,
        "ID": nombre,
        "Categoría": "Planeta",
        "Masa": datos_tecnicos["Masa"],
        "Gravedad": datos_tecnicos["Gravedad"],
        "Diámetro": datos_tecnicos["Diámetro"],
        "Contenido": resumen,
        "URL_Wikipedia": url
    }
lista_final = []
for nombre, url in enlaces_planetas.items():
    datos = extraer_datos_planeta(nombre, url)
    lista_final.append(datos)

df = pd.DataFrame(lista_final)

# Reordenar
columnas = ['ID', 'Nombre', 'Categoría', 'Masa', 'Gravedad', 'Diámetro', 'Contenido', 'URL_Wikipedia']
df = df[columnas]

df.to_csv('planetas_completos.csv', index=False, encoding='utf-8-sig')
print("Porceso finalizado exitosamente")

Porceso finalizado exitosamente


Repetimos el proceso con los planetas enanos del sistema solar

In [ ]:
def arreglar(celda): # Arregla problemas con la notacion científica
    for sup in celda.find_all('sup'):
        contenido_sup = sup.get_text().strip()
        if contenido_sup.isdigit():
            sup.replace_with(f"E{contenido_sup}")

    texto = celda.get_text()
    texto = re.sub(r'[×·]\s*10\s*E', 'E', texto)
    return texto

def limpiar(texto):
    texto = str(texto)
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.replace('\xa0', ' ')
    return texto.strip()

# Como la tabal anexo de la wikipedia es dificil de leer y son pocos planetas enanos, metemos los enlaces a mano
enlaces_planetas = {
    "Ceres": "https://es.wikipedia.org/wiki/Ceres_(planeta_enano)",
    "Eris": "https://es.wikipedia.org/wiki/Eris_(planeta_enano)",
    "Plutón": "https://es.wikipedia.org/wiki/Plut%C3%B3n_(planeta_enano)",
    "Makemake": "https://es.wikipedia.org/wiki/Makemake_(planeta_enano)",
    "Haumea": "https://es.wikipedia.org/wiki/Haumea_(planeta_enano)",
}

def extraer_datos_planeta(nombre, url):
    resumen = "Descripción no disponible."
    datos_tecnicos = {"Masa": "n/a", "Gravedad": "n/a", "Diámetro": "n/a"}

    try:
        r = requests.get(url, timeout=10)
        time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, 'html.parser')

            # Extraer el Contenido (Primer párrafo largo)
            for p in soup.find_all('p'):
                texto_p = p.get_text().strip()
                if len(texto_p) > 120:
                    resumen = limpiar(texto_p)
                    break

            # Extraer Datos Técnicos de la 'infobox' que es la ficha lateral de las páginas de la wiki
            infobox = soup.find('table', class_='infobox')
            if infobox:
                filas_info = infobox.find_all('tr')
                for f in filas_info:
                    texto_fila = f.get_text().lower()

                    if 'masa' in texto_fila and datos_tecnicos["Masa"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Masa"] = limpiar(arreglar(valor))

                    if 'gravedad' in texto_fila and datos_tecnicos["Gravedad"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Gravedad"] = limpiar(arreglar(valor))

                    if 'diámetro' in texto_fila and datos_tecnicos["Diámetro"] == "n/a":
                        valor = f.find('td')
                        if valor: datos_tecnicos["Diámetro"] = limpiar(arreglar(valor))


    except Exception as e:
        print(f"Error con {nombre}: {e}")

    return {
        "Nombre": nombre,
        "ID": nombre,
        "Categoría": "Plantea Enano",
        "Masa": datos_tecnicos["Masa"],
        "Gravedad": datos_tecnicos["Gravedad"],
        "Diámetro": datos_tecnicos["Diámetro"],
        "Contenido": resumen,
        "URL_Wikipedia": url
    }
lista_final = []
for nombre, url in enlaces_planetas.items():
    datos = extraer_datos_planeta(nombre, url)
    lista_final.append(datos)

df = pd.DataFrame(lista_final)

# Reordenar
columnas = ['ID', 'Nombre', 'Categoría', 'Masa', 'Gravedad', 'Diámetro', 'Contenido', 'URL_Wikipedia']
df = df[columnas]

df.to_csv('planetas_enanos_completos.csv', index=False, encoding='utf-8-sig')
print("Porceso finalizado exitosamente")

Porceso finalizado exitosamente


Ahora generamos la tabla de las galaxias

In [ ]:
def limpiar(texto):
    texto = str(texto)
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.replace('\xa0', ' ')
    return texto.strip()

url_galaxias = "https://es.wikipedia.org/wiki/Anexo:Galaxias"

respuesta = requests.get(url_galaxias)
time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
datos_lista = []

if respuesta.status_code == 200:
    soup = BeautifulSoup(respuesta.text, 'html.parser')
    tabla = soup.find('table', class_='wikitable')
    filas = tabla.find_all('tr')[1:]

    for fila in filas:
        celdas = fila.find_all(['td', 'th'])
        if len(celdas) >= 4:
            nombre = limpiar(celdas[0].get_text())
            tipo = limpiar(celdas[2].get_text())
            constelacion = limpiar(celdas[3].get_text())

            tag_a = celdas[0].find('a', href=re.compile(r'^/wiki/'))
            url_g = "https://es.wikipedia.org" + tag_a['href'] if tag_a else "Sin enlace"

            datos_lista.append({
                'Nombre': nombre,
                'ID': nombre,
                'Constelacion': constelacion,
                'URL_Wikipedia': url_g,
                'Categoría': 'Galaxia'
            })

    df = pd.DataFrame(datos_lista)

    def contexto_galaxia(fila):
        resumen = "Descripción no disponible."
        distancia = "n/a"
        if fila['URL_Wikipedia'] != "Sin enlace":
            try:
                r = requests.get(fila['URL_Wikipedia'], timeout=5)
                time.sleep(5) # Tiempo de espera para no saturar la Wikipedia
                s = BeautifulSoup(r.text, 'html.parser')

                for p in s.find_all('p'):
                    txt = p.get_text().strip()
                    if len(txt) > 120:
                        resumen = limpiar(txt)
                        break

                infobox = s.find('table', class_='infobox')
                if infobox:
                    for tr in infobox.find_all('tr'):
                        if 'Distancia' in tr.get_text().lower():
                            val = tr.find('td')
                            if val: distancia = limpiar(val.get_text())
            except: pass
        return pd.Series([resumen, distancia])

    df[['Contenido', 'Distancia']] = df.apply(contexto_galaxia, axis=1)

    # Reordenar
    columnas_finales = [
    'ID', 'Nombre', 'Categoría', 'Constelacion', 'Contenido', 'Distancia', 'URL_Wikipedia'
    ]
    df = df[columnas_finales]

    df.to_csv('galaxias_completo.csv', index=False, encoding='utf-8-sig')
    print("Porceso finalizado exitosamente")

Porceso finalizado exitosamente


## Juntar todas las tablas

In [ ]:
df_constelaciones = pd.read_csv('constelaciones_completa.csv')
df_estrellas = pd.read_csv('estrellas_completo.csv')
df_galaxias = pd.read_csv('galaxias_completo.csv')
df_planetas = pd.read_csv('planetas_completos.csv')
df_enanos = pd.read_csv('planetas_enanos_completos.csv')

universo_df = pd.concat([
    df_constelaciones,
    df_estrellas,
    df_galaxias,
    df_planetas,
    df_enanos
], ignore_index=True, sort=False)

universo_df = universo_df.fillna("n/a") # Rellenamos huecos con n/a si esa columna no está con info

# Reordenar
columnas_finales = [
'ID', 'Nombre', 'Nombre_Latin', 'Categoría', 'Constelacion', 'Brillo', 'Coordenadas', 'Ubicacion', 'Distancia', 'Masa', 'Gravedad', 'Diámetro', 'Contenido', 'URL_Wikipedia'
]
universo_df = universo_df[columnas_finales]

# Guardar
universo_df.to_csv('base_de_datos_final.csv', index=False, encoding='utf-8-sig')
print("Se ha guardado el archivo =)")

Se ha guardado el archivo =)
